# Data preprocessing

In [1]:
import numpy as np
import uuid
from tqdm.auto import tqdm

In [2]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 83.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import re
import fitz  # PyMuPDF

def clean_text(t):
    t = re.sub(r'-\n', '', t) 
    t = re.sub(r'\s+\n', '\n', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'Page \d+', '', t)
    return t.strip()

doc = fitz.open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/book.pdf")
pages = []
page_start_positions = []  # сохраняем, где начинается каждая страница в общем тексте
text = ""
for i, page in enumerate(doc):
    page_start_positions.append(len(text))
    page_text = page.get_text("text")
    page_text = clean_text(page_text)
    text += page_text + "\n"
    pages.append(page_text)

with open("textbook.txt", "w", encoding="utf-8") as f:
    f.write(text)

print(f"Извлечено {len(pages)} страниц. Общая длина текста: {len(text):,} символов.")


Извлечено 753 страниц. Общая длина текста: 2,254,755 символов.


In [4]:
# print(text[:100000])

# Model

In [4]:
from transformers import pipeline
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)


model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

2025-11-10 07:13:35.131498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762758815.309456      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762758815.358486      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)

print(generation_pipeline(messages, max_new_tokens=256, do_sample=True, temperature=0.3, top_p=0.9)[0]['generated_text'][-1]['content'])

Device set to use cuda:0


A large language model is an artificial intelligence system that can generate human-like text based on the input it receives. These models use deep learning techniques, such as neural networks and recurrent neural networks (RNNs), to analyze vast amounts of data and learn patterns in natural language.

The most well-known example of a large language model is GPT-3, developed by OpenAI. It has been trained on over 1 trillion words from books, articles, and other sources, making it one of the largest and most powerful AI systems ever created.

Large language models have many potential applications, including language translation, chatbots, content generation, and even creative writing. They can also be used for tasks like sentiment analysis, where they can identify emotional tone in text or predict future trends based on historical data.

However, there are also concerns about the ethical implications of these models, particularly around issues related to bias and privacy. As with any ne

# vector bd

In [6]:
# import torch.nn.functional as F
# from transformers import AutoModel

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large", model_kwargs={'torch_dtype': torch.float16})

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [7]:
!pip install langchain-qdrant

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 13.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 22.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.72
    Uninstalling langchain-core-0.3.72:
      Successfully uninstalled langchain-core-0.3.72
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.
langchain-text-splitters 0.3.9 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.


In [8]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="psycology_e5",
    on_disk_payload=True,
    vectors_config=models.VectorParams(
        size=1024,
        distance=models.Distance.COSINE,
        on_disk=True
    ),
)

True

# chunk-split

In [9]:
!pip install langchain-community

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.9
    Uninstalling langchain-text-splitters-0.3.9:
      Successfully uninstalled langchain-text-splitters-0.3.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.4 which is incompatible.
langchain 0.3.27 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.0.0 which is incompatible.


In [10]:
def find_page_for_chunk(start_index, page_starts):
    """Находит, на какой странице начинается чанк по его позиции в тексте."""
    if start_index is None or start_index < 0:
        return 1  # fallback — если не нашли позицию, считаем первой страницей
    for i, p_start in enumerate(page_starts):
        if start_index < p_start:
            return max(1, i)  # страницы нумеруются с 1
    return len(page_starts) 

In [41]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=80, separators=["\n\n", "\n", " ", ""])

In [42]:
chunks = []
last_pos = 0
for chunk in text_splitter.split_text(text):
    if not chunk.strip():  # пропускаем пустые чанки
        continue

    start_index = text.find(chunk, last_pos)
    if start_index == -1:
        # если find не нашёл (например, повтор текста), пробуем искать с начала
        start_index = text.find(chunk)
        if start_index == -1:
            # если вообще не нашли — пропускаем этот чанк
            print(f"Не найден чанк{chunk} в тексте, пропущен.")
            continue

    page = find_page_for_chunk(start_index, page_start_positions)
    chunks.append({"text": chunk, "page": page})
    last_pos = start_index + len(chunk)

print(f"Получено {len(chunks)} чанков.")

Получено 2217 чанков.


In [43]:
vectors = embedding_model.encode([chunk["text"] for chunk in chunks],
                                 batch_size=32, device=device, normalize_embeddings=True, show_progress_bar=True).tolist()

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

In [44]:
batch_size = 64

for i in tqdm(range(0, len(vectors), batch_size)):
    batch_points = [
        models.PointStruct(
            id=str(uuid.uuid4()),
            vector=vectors[j],
            payload={
                'text': chunks[j]["text"],
                'page': chunks[j]["page"],
            }
        )
        for j in range(i, min(i + batch_size, len(vectors)))
    ]
    client.upsert(collection_name='psycology_e5', points=batch_points)


  0%|          | 0/35 [00:00<?, ?it/s]

# Submission

In [51]:
def semantic_search(client, query, limit=3, collection_name="psycology_e5"):
    """
    Выполняет семантический поиск в коллекции Qdrant.

    Аргументы:
        client: экземпляр QdrantClient
        query: текстовый запрос пользователя
        limit: количество возвращаемых чанков (по умолчанию 10)
        collection_name: имя коллекции (по умолчанию 'psycology_e5')

    Возвращает:
        Список словарей формата:
        [
            {
                "text": "...",          # сам чанк
                "page": 123,            # страница, если есть в payload
                "score": 0.87           # косинусная близость
            },
            ...
        ]
    """
    # Кодируем запрос
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        device=device
    ).tolist()

    # Делаем запрос в Qdrant
    hits = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=limit,
        score_threshold=0.2
    )

    # Безопасно обрабатываем результаты
    if not hits:
        print("Предупреждение: по запросу ничего не найдено.")
        return []

    # Собираем полезную информацию из результатов
    results = []
    for hit in hits:
        payload = hit.payload or {}
        results.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "score": hit.score
        })

    return results


In [52]:
def llm_answer(query, context):
    prompt = f"""Текст из учебника психологии:
{context}

Вопрос:
{query}"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a very skeptical scientist in the field of psychology. You will receive a context consisting of text clippings from a book on the desired topic. Your task is to answer the user as accurately and honestly as possible. Make sure that the answer is detailed, specific, and directly related to the question. Do not add information that is not directly supported by the provided clippings from the book. If there is no direct answer in the text, tell me about it honestly."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    output = generation_pipeline(
        messages,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
    )

    # Проверяем структуру вывода, т.к. она может отличаться между версиями Transformers
    if isinstance(output[0]["generated_text"], list):
        # новый формат: список сообщений
        return output[0]["generated_text"][-1]["content"]
    elif isinstance(output[0]["generated_text"], str):
        # старый формат: просто строка
        return output[0]["generated_text"]
    else:
        # fallback
        return str(output[0])


In [53]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query, limit=5)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    # pages = sorted(list({chunk["page"] for chunk in relevant_chunks}))
    references = json.dumps({"pages": pages})

    answer = llm_answer(query, context)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df = pd.DataFrame(results)
df.to_csv("submission_basic_rag9.csv", index=False)
print("submission_basic_rag9.csv saved")


  0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_48/2959729219.py:30: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


submission_basic_rag9.csv saved


# Не по лимиту, а по уверенности выбираем количество чанков для формирования ответа

In [60]:
from sentence_transformers import CrossEncoder

# Загружаем легкую и быструю модель для оценки релевантности
scorer = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=device)

def semantic_search(
    client,
    query,
    collection_name="psycology_e5",
    top_k=15,
    confidence_threshold=0.6,
    show_confidences=True
):
    # 1️⃣ Получаем top_k ближайших по вектору
    query_vector = embedding_model.encode(query, normalize_embeddings=True, device=device).tolist()
    res = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        score_threshold=0.2
    )
    hits = res.points if hasattr(res, "points") else res[0]

    if not hits:
        print("⚠️ По запросу ничего не найдено.")
        return []

    candidates = []
    for hit in hits:
        payload = getattr(hit, "payload", None) or hit.get("payload", {})
        candidates.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "score": hit.score if hasattr(hit, "score") else hit.get("score")
        })

    # 2️⃣ Используем cross-encoder для оценки уверенности
    pairs = [(query, c["text"]) for c in candidates]
    confidences = scorer.predict(pairs)
    for c, conf in zip(candidates, confidences):
        c["confidence"] = float(conf)

    # 3️⃣ Сортируем и фильтруем
    sorted_candidates = sorted(candidates, key=lambda x: x["confidence"], reverse=True)
    filtered = [c for c in sorted_candidates if c["confidence"] >= confidence_threshold]

    if show_confidences:
        print(f"\nЗапрос: {query}")
        for c in sorted_candidates:
            clean_text = c["text"].replace("\n", " ")[:80]
            print(f"Уверенность {c['confidence']:.2f} | {clean_text}...")

    return filtered


In [ ]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset2/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query, confidence_threshold=0.6)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    # pages = sorted(list({chunk["page"] for chunk in relevant_chunks}))
    references = json.dumps({"pages": pages})

    answer = llm_answer(query, context)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df = pd.DataFrame(results)
df.to_csv("submission_basic_rag10.csv", index=False)
print("submission_basic_rag10.csv saved")


  0%|          | 0/50 [00:00<?, ?it/s]


Запрос: What is the scientific method in psychology?
Уверенность 1.00 | education institutions, increasing the number of Black Americans who went on to ...
Уверенность 1.00 | psychological science is empirical, based on measurable data. In general, scienc...
Уверенность 1.00 | education institutions, increasing the number of Black Americans who went on to ...
Уверенность 1.00 | respectively. These developments provided an opportunity for Indian researchers ...
Уверенность 1.00 | respectively. These developments provided an opportunity for Indian researchers ...
Уверенность 1.00 | against the world, but they tend to be too complex to be tested all at once; ins...
Уверенность 1.00 | it is happy is not a hypothesis that can be tested since we have no way to measu...
Уверенность 1.00 | 1 Introduction to Psychology 2002). Nash was the subject of the 2001 movie A Bea...
Уверенность 1.00 | 1 Introduction to Psychology 2002). Nash was the subject of the 2001 movie A Bea...
Уверенность 1.00 | 